# Collections Recovery — Forensic Analysis
**Leadership claim:** *"Recovery improved 11% month-on-month."* The team is not convinced.
This notebook rebuilds the numbers from raw data and shows the reasoning at each step.

**Headline findings**
1. Recovery is **flat**, not +11%. The "+11%" is a cherry-picked Feb->Mar pair.
2. Collections **activity does not measurably drive recovery** (no dose-response).
3. Real data-quality traps had to be fixed first (duplicate payments, reference reuse, agent identity, timezones, disposition synonyms).

Transparent, reproducible methods preferred over black-box models.

In [1]:
import pandas as pd, numpy as np
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
RAW='../../data'   # adjust to your raw path (e.g. /mnt/user-data/uploads)
pd.options.display.float_format=lambda x:f'{x:,.2f}'

## 1. Payments: two duplication problems, opposite treatments

In [2]:
p=pd.read_csv(f'{RAW}/payments.csv',parse_dates=['event_at'])
print('rows',len(p),'| unique payment_id',p.payment_id.nunique(),
      '| unique payment_reference',p.payment_reference.nunique())
print(p.payment_status.value_counts())
print('exact dup payment_id rows:', p.duplicated("payment_id").sum())
reuse=p.groupby('payment_reference').account_id.nunique()
print('references used on >1 account:',(reuse>1).sum())

rows 25500 | unique payment_id 25000 | unique payment_reference 20821
payment_status
SUCCESS     17880
FAILED       3744
PENDING      2592
REVERSED     1284
Name: count, dtype: int64
exact dup payment_id rows: 500
references used on >1 account: 3407


**Decision:** de-dup on `payment_id`; **never** on `payment_reference`. Recovery = `SUCCESS` only.

In [3]:
p=p.drop_duplicates('payment_id')
s=p[p.payment_status=='SUCCESS'].copy()
print(f'clean recovery: Rs {s.amount.sum():,.0f}')
naive=s.sort_values('event_at').drop_duplicates('payment_reference').amount.sum()
print(f'naive ref-dedup (WRONG) would erase {(1-naive/s.amount.sum())*100:.1f}% of recovery')

clean recovery: Rs 1,315,583,965
naive ref-dedup (WRONG) would erase 12.6% of recovery


## 2. Testing the "+11% MoM" claim

In [4]:
s['ym']=s.event_at.dt.to_period('M')
m=s.groupby('ym').amount.sum().to_frame('recovery'); m['mom_%']=m.recovery.pct_change()*100
display(m)
print('Jan vs Jul: %.2f%%'%((m.recovery.loc['2026-07']/m.recovery.loc['2026-01']-1)*100))
full=m[m.index.astype(str)<'2026-08']
print('full-month CV: %.1f%%'%(full.recovery.std()/full.recovery.mean()*100))

,recovery,mom_%
ym,,
2026-01,"187,229,127.59",NaN
2026-02,"170,142,453.76",-9.13
2026-03,"188,912,374.02",11.03
2026-04,"175,138,043.41",-7.29
2026-05,"184,250,278.50",5.20
2026-06,"175,559,726.97",-4.72
2026-07,"187,242,265.08",6.65
2026-08,"47,109,695.31",-74.84


Jan vs Jul: 0.01%
full-month CV: 4.1%


Only **Feb->Mar = +11%**; Jan≈Jul (+0.01%); CV≈4% -> flat with noise. **The reported improvement is a cherry-pick.** (Aug is a partial month.)

## 3. Does activity drive recovery? (decisive causal-null test)

In [5]:
acc=pd.read_csv(f'{RAW}/accounts.csv')[['account_id','risk_segment','dpd']]
paid=set(s.account_id); acc['paid']=acc.account_id.isin(paid).astype(int)
ans=set(pd.read_csv(f'{RAW}/calls.csv',usecols=['account_id','call_status'])
        .query("call_status=='ANSWERED'").account_id)
acc['rpc']=acc.account_id.isin(ans).astype(int)
att=pd.read_csv(f'{RAW}/call_attempts.csv',usecols=['account_id']).groupby('account_id').size()
acc['att']=acc.account_id.map(att).fillna(0)
print('pay rate by RPC:'); print(acc.groupby('rpc').paid.mean())
print('RPC chi2 p=%.3f'%stats.chi2_contingency(pd.crosstab(acc.rpc,acc.paid))[1])
print('attempts~paid r=%.4f p=%.3f'%stats.pointbiserialr(acc.paid,acc.att))
acc['bin']=pd.cut(acc.att,[-1,0,2,5,10,999],labels=['0','1-2','3-5','6-10','10+'])
print('pay rate by attempt dose:'); display(acc.groupby('bin').paid.mean())

pay rate by RPC:
rpc
0   0.44
1   0.45
Name: paid, dtype: float64
RPC chi2 p=0.189
attempts~paid r=0.0060 p=0.299
pay rate by attempt dose:


bin
0      0.44
1-2    0.44
3-5    0.44
6-10   0.45
10+    0.37
Name: paid, dtype: float64

**No effect, no dose-response** (RPC p=0.19; attempts p=0.30). Recovery is **borrower-driven / exogenous**; campaign/channel/agent attribution is correlation with *who was targeted*, not causation.

## 4. Other forensics (summary)
- **Agents:** 30k rows -> 1k real `agent_id`; `employee_code` maps to ~30 agents -> tenure analysis unreliable.
- **Timezones:** `event_at` naive; mean call hour ~11.5 across all 3 zones -> intraday analysis invalid.
- **Dispositions:** `PTP` == `PROMISE_TO_PAY` synonyms across versions -> unify.
- **Portfolio mix / denominator:** stable -> *not* drivers.

## 5. Investment implication
Activity != recovery, so "more of the same" ≈ Rs 0 incremental. Recommend a **randomized holdout** (~3%, 10-12 wks, Rs 30-40 L) to measure true lift; if forced to allocate, choose **AI-voice/digital automation as a cost play**. Full reasoning in `reports/executive_memo.md`.